# Reference Frames And Transforms

This notebook explains the first semantic layer of PyTex: named frames, explicit
domains, and reusable rigid transforms. The goal is not only to show the API, but
to fix the meaning of every vector before later orientation, EBSD, or diffraction
workflows are attempted.

## Learning Goals

- construct `ReferenceFrame` objects and understand frame domains
- express rigid mappings with `FrameTransform`
- work with `VectorSet` so frame meaning stays attached to batched data
- connect the code to the mathematical map

## Mathematical Contract

PyTex treats a frame transform as

$$
\mathbf{v}^{(t)} = \mathbf{R}_{s\rightarrow t}\,\mathbf{v}^{(s)} + \mathbf{t}_{s\rightarrow t},
$$

where the source and target frames are explicit objects, not comments or naming
conventions.


In [ ]:
from pathlib import Path
import tempfile

import numpy as np

from pytex import (
    AcquisitionGeometry,
    AtomicSite,
    BenchmarkManifest,
    build_crystal_scene,
    CalibrationRecord,
    CrystalCellOverlay,
    CrystalDirection,
    CrystalDirectionOverlay,
    CrystalMap,
    CrystalPlane,
    CrystalPlaneOverlay,
    DirectionAnnotationStyle,
    DiffractionGeometry,
    EulerSet,
    ExperimentManifest,
    FrameDomain,
    FrameTransform,
    Handedness,
    InversePoleFigure,
    KernelSpec,
    KinematicSimulation,
    Lattice,
    get_phase_fixture,
    list_phase_fixtures,
    list_style_themes,
    MeasurementQuality,
    MillerIndex,
    ODF,
    Orientation,
    OrientationRelationship,
    OrientationSet,
    Phase,
    PhaseTransformationRecord,
    PoleFigure,
    PowderPattern,
    PowderReflection,
    ReferenceFrame,
    read_validation_manifest,
    read_workflow_result_manifest,
    resolve_style,
    RadiationSpec,
    Rotation,
    ScatteringSetup,
    SymmetrySpec,
    TransformationVariant,
    UnitCell,
    ValidationManifest,
    VectorSet,
    WorkflowResultManifest,
    ZoneAxis,
    PlaneAnnotationStyle,
    generate_saed_pattern,
    generate_xrd_pattern,
    plot_odf,
    plot_crystal_structure_3d,
    plot_inverse_pole_figure,
    plot_orientations,
    plot_pole_figure,
    plot_saed_pattern,
    plot_symmetry_elements,
    plot_symmetry_orbit,
    plot_vector_set,
    plot_xrd_pattern,
)


def make_crystal_frame():
    return ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )


def make_context():
    crystal = make_crystal_frame()
    specimen = ReferenceFrame(
        "specimen",
        FrameDomain.SPECIMEN,
        ("x", "y", "z"),
        Handedness.RIGHT,
    )
    map_frame = ReferenceFrame(
        "map",
        FrameDomain.MAP,
        ("i", "j", "k"),
        Handedness.RIGHT,
    )
    detector = ReferenceFrame(
        "detector",
        FrameDomain.DETECTOR,
        ("u", "v", "n"),
        Handedness.RIGHT,
    )
    lab = ReferenceFrame(
        "lab",
        FrameDomain.LABORATORY,
        ("X", "Y", "Z"),
        Handedness.RIGHT,
    )
    phase = get_phase_fixture("ni_fcc").load_phase(crystal_frame=crystal)
    return crystal, specimen, map_frame, detector, lab, phase


def describe_phase_fixture(fixture_id):
    record = get_phase_fixture(fixture_id)
    return {
        "fixture_id": record.fixture_id,
        "display_name": record.display_name,
        "artifact_path": str(record.artifact_path),
        "metadata_path": str(record.metadata_path),
        "intended_uses": tuple(record.metadata["intended_uses"]),
    }


def load_zr_hcp_phase():
    return get_phase_fixture("zr_hcp").load_phase(crystal_frame=make_crystal_frame())


def load_diamond_phase():
    return get_phase_fixture("diamond").load_phase(crystal_frame=make_crystal_frame())


def publication_crystal_style():
    return {
        "crystal": {
            "atom_radius_scale": 0.5,
            "atom_edgewidth": 0.0,
            "atom_surface_resolution": 34,
            "bond_surface_resolution": 28,
            "bond_alpha": 0.72,
            "bond_color": "#7c8ea3",
            "atom_specular_strength": 0.42,
            "light_specular": 0.4,
        }
    }


In [ ]:
crystal, specimen, map_frame, detector, lab, phase = make_context()

specimen_to_map = FrameTransform(
    source=specimen,
    target=map_frame,
    rotation_matrix=np.eye(3),
    translation_vector=np.array([0.0, 0.0, 0.0]),
)

vectors = VectorSet(
    values=np.array(
        [
            [1.0, 0.0, 0.0],
            [0.0, 1.0, 0.0],
            [0.0, 0.0, 1.0],
        ]
    ),
    reference_frame=specimen,
)

mapped = specimen_to_map.apply_to_vectors(vectors)

print(vectors.reference_frame.name, "->", mapped.reference_frame.name)
print(mapped.values)


## Why The `VectorSet` Matters

A raw `(n, 3)` array is not enough when the frame matters. PyTex therefore treats
batched vectors as a first-class semantic object:

- the storage remains NumPy-backed and vectorized
- the shared frame is explicit
- later transforms can reject mismatches early

This is the pattern that the rest of the library follows: preserve vectorization,
but never let the science collapse back into anonymous arrays.


In [ ]:
try:
    wrong_vectors = VectorSet(values=np.array([[1.0, 0.0, 0.0]]), reference_frame=crystal)
    specimen_to_map.apply_to_vectors(wrong_vectors)
except ValueError as exc:
    print(type(exc).__name__)
    print(exc)
